# Single-model evaluation

`RUNS_ROOT` defaults to the conventional processed-data `TASK/runs` directory, so normal use requires changing only `RUN_NAME`. `RUN_NAME` is the current directory leaf or storage alias, the immutable scientific run name is read separately from the bundle. Change `RUNS_ROOT` independently for moved collections, or point it at an Optuna study's `trials/` directory.

Valid existing artifacts are loaded without model reconstruction or inference. Missing selected roles are generated locally on CPU by default. Partial, corrupt, stale, or incompatible artifacts fail unless `REBUILD_INCOMPATIBLE_ARTIFACTS` is deliberately enabled. Local notebook generation never contacts W&B. Terminal non-completed runs are labelled provisional and always use their validated `best_checkpoint.pt`.

The authoritative `normalized_group_macro_rmse` is dimensionless, lower is better, and is not a percentage.

In [ ]:
from pathlib import Path

from IPython.display import display as show

from src import analysis, common

evaluation_workflow = analysis.evaluation.workflow

TASK = "steady_flow"
RUNS_ROOT: Path = common.paths.resolve_runs_root(TASK)
RUN_NAME = "replace_with_current_run_name"
RUN_LABEL = "Run A"

# RUN_NAME selects the current directory leaf, scientific identity comes from the saved run bundle.
RUN_DIR = RUNS_ROOT / RUN_NAME

AUTO_BUILD_MISSING_ARTIFACTS = True
ARTIFACT_DEVICE = "cpu"
REBUILD_INCOMPATIBLE_ARTIFACTS = False
ARTIFACT_ROLES = ("id", "ood")

In [ ]:
context_specs = (
    evaluation_workflow.EvaluationContextSpec(key="validation_dataset", label="ID", artifact_role="id"),
    evaluation_workflow.EvaluationContextSpec(key="shifted_dataset", label="OOD", artifact_role="ood"),
)
evaluation = evaluation_workflow.prepare_evaluation_workflow(
    (evaluation_workflow.EvaluationRunSelection(run_dir=RUN_DIR, label=RUN_LABEL),),
    context_specs,
    artifact_roles=ARTIFACT_ROLES,
    auto_build_missing=AUTO_BUILD_MISSING_ARTIFACTS,
    rebuild_incompatible=REBUILD_INCOMPATIBLE_ARTIFACTS,
    device_policy=ARTIFACT_DEVICE,
)
print(list(evaluation.report))

In [ ]:
show(evaluation.panel)